# How to Use the WaRP-C Data Pipeline
El Mehdi Ziate 2026-03-29

## Step 0 — Imports (same for every notebook)

In [ ]:
import sys 
from pathlib import Path
import torch
import torch.nn as nn

# auto-detect project root
root = Path.cwd()
while not (root / 'Pipeline_').exists() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from Pipeline_.preprocessor import WaRPPreprocessor, WARP_MEAN, WARP_STD

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


## Step 1 — Create the preprocessor

One object. Same for everyone. Just change `model_type` to match your model.

In [3]:
pp = WaRPPreprocessor(
    raw_root       = root / 'Dataset/raw/Warp-C',
    processed_root = root / 'Dataset/processed',
    stats_file     = root / 'Dataset/dataset_stats.json',
    img_size       = 224,
    batch_size     = 32,
    num_workers    = 4,
    seed           = 42,
)

pp.summary()


  WaRPPreprocessor — Pipeline Summary
  Input size      : 224×224 px
  Mean (R,G,B)    : [0.337, 0.344, 0.35]  [WaRP-C stats]
  Std  (R,G,B)    : [0.216, 0.209, 0.218]
  Batch size      : 32
  Minority thresh : < 30% of class mean

  Train pipeline  :
    PadToSquare(reflect) → RandomResizedCrop(224)
    → Flips → Rotation → ColorJitter(b=0.4) → Blur → Normalize → Erase

  Val/Test pipeline:
    PadToSquare(reflect) → Resize(224) → Normalize

  Imbalance strategy (ρ=59.67, Buda et al. 2018):
    Layer 1 — WeightedRandomSampler
    Layer 2 — CrossEntropyLoss(weight=class_weights)
    Layer 3 — Stronger augmentation for minority classes

  Leakage fix     : 18 duplicate filenames removed from train


## Step 2 — Clean the dataset (run ONCE, then never again)

`prepare()` does three things:
1. Finds the 18 duplicate filenames that appear in both train and test (data leakage)
2. Removes them from train only 
3. Copies everything to `Dataset/processed/` in a clean folder structure

**Output on disk:** original-size cleaned images. NOT 224×224. NOT normalised.  
**The DataLoader handles resizing and normalisation in RAM during training.**

In [ ]:
pp.prepare(force=False)   
# force=False → skips if processed/ already exists (safe to re-run)
# force=True  → deletes and recreates from scratch

## Step 3 — Get your DataLoaders

**This is the only line that changes between teammates.**  

### What each model_type does under the hood

| model_type | Sampler | Minority aug | MixUp | Crop scale |
|---|---|---|---|---|
| `"resnet50"` | ON | ON | OFF | 0.6–1.0 |
| `"efficientnet"` | ON | ON | ON | 0.6–1.0 |
| `"swin"` | ON | ON | ON | 0.6–1.0 |
| `"vit"` | ON | ON | ON | 0.6–1.0 |
| `"convnext"` | ON | ON | ON | 0.6–1.0 |
| `"edgevit"` | ON | ON | OFF | **0.7**–1.0 |
| `"mobilevit"` | ON | ON | OFF | **0.7**–1.0 |
| `"llava"` | OFF | OFF | OFF | val pipeline only |
| `"gnn"` | OFF | OFF | OFF | val pipeline only |

EdgeViT and MobileViT use a gentler crop (0.7 instead of 0.6) because they are lightweight models that struggle with very aggressive zoom augmentation.  
LLaVA and GNN get the val pipeline because they are not trained — LLaVA is prompted, GNN uses features.

In [4]:
# Change this one string to match your model
MODEL_TYPE = 'resnet50'   # or 'efficientnet', 'swin', 'convnext', 'edgevit', etc.


train_loader, test_loader = pp.get_loaders(model_type=MODEL_TYPE)

print(f'\nTrain batches : {len(train_loader)}')
print(f'Test  batches : {len(test_loader)}')

[get_loaders] model='resnet50'  sampler=True  minority_aug=True  mixup=False  crop_min=0.6  inference_only=False
[DataLoaders]  train=273 batches  test=49 batches  sampler=WeightedRandom

Train batches : 273
Test  batches : 49


## Step 4 — Set up the loss function with class weights

This is the second layer of imbalance correction.  
The sampler (Layer 1) controls how often each class appears in a batch.  
The weighted loss (Layer 2) controls how hard the model tries on each class when it does appear.  

`bottle-oil-full` (24 images) gets a weight ~59× higher than `bottle-transp` (1432 images).

In [5]:
class_weights = pp.get_class_weights(device=DEVICE)
criterion     = nn.CrossEntropyLoss(weight=class_weights)

print(f'Class weights shape: {class_weights.shape}')   # (28,)
print(f'Min weight: {class_weights.min():.4f}  (majority class — bottle-transp)')
print(f'Max weight: {class_weights.max():.4f}  (minority class — bottle-oil-full)')
print(f'Ratio: {class_weights.max()/class_weights.min():.1f}x')

Class weights shape: torch.Size([28])
Min weight: 0.1125  (majority class — bottle-transp)
Max weight: 6.7108  (minority class — bottle-oil-full)
Ratio: 59.7x


## Step 5 — Verify a batch looks correct

In [6]:
images, labels = next(iter(train_loader))

print('Batch verification')
print(f'  images.shape  : {images.shape}')          # (32, 3, 224, 224)
print(f'  labels.shape  : {labels.shape}')          # (32,)
print(f'  pixel range   : [{images.min():.3f}, {images.max():.3f}]')  # normalised — around [-2, 2]
print(f'  unique classes in batch: {len(labels.unique())} / 28')

# if unique classes is close to 28, the sampler is working correctly
# without sampler most batches would have 0 bottle-oil-full images

Batch verification
  images.shape  : torch.Size([32, 3, 224, 224])
  labels.shape  : torch.Size([32])
  pixel range   : [-4.460, 4.547]
  unique classes in batch: 17 / 28


## Step 6 — The training loop

The only difference between models is whether MixUp is applied.  
The `_use_mixup` flag is set automatically by `get_loaders(model_type=...)`  
so you do not need to remember which model uses it.

In [ ]:
# ── Template training loop — copy this into your model notebook ──────────────

# model = YourModel().to(DEVICE)
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# for epoch in range(NUM_EPOCHS):
#     model.train()
#     for images, labels in train_loader:
#         images, labels = images.to(DEVICE), labels.to(DEVICE)
#
#         # MixUp — only runs for models that need it (set by model_type)
#         if getattr(pp, '_use_mixup', False):
#             images, labels = WaRPPreprocessor.mixup_batch(images, labels, num_classes=28)
#
#         optimizer.zero_grad()
#         loss = criterion(model(images), labels)
#         loss.backward()
#         optimizer.step()

print('Training loop template ready — uncomment and add your model above')

## Step 7 — TTA at final evaluation

Test-Time Augmentation. Use this ONLY at the end for your best final number — not during training validation.  
Runs the same image through 5 different views, averages the softmax scores.  
Typical gain: +1–3% accuracy at zero training cost.

In [ ]:
# ── TTA evaluation template ───────────────────────────────────────────────────

# tta_pipelines = pp.get_tta_transforms()   # list of 5 deterministic transforms
#
# model.eval()
# all_preds = []
#
# with torch.no_grad():
#     for img_path, label in test_samples:
#         img = Image.open(img_path).convert('RGB')
#         probs = []
#         for tfm in tta_pipelines:
#             tensor = tfm(img).unsqueeze(0).to(DEVICE)
#             probs.append(model(tensor).softmax(dim=1))
#         final_pred = torch.stack(probs).mean(0).argmax()
#         all_preds.append(final_pred)

print('TTA template ready')

## Quick reference — full usage in 8 lines

In [ ]:
# ── Everything you need, in 8 lines ──────────────────────────────────────────

from Pipeline_.preprocessor import WaRPPreprocessor
import torch.nn as nn

pp                        = WaRPPreprocessor()
pp.prepare()                                           # run once only
train_loader, test_loader = pp.get_loaders(model_type='resnet50')  # change this
criterion                 = nn.CrossEntropyLoss(weight=pp.get_class_weights(device=DEVICE))

# images, labels = next(iter(train_loader))
# images: (32, 3, 224, 224) float tensor, normalised
# labels: (32,) int tensor, values 0-27
# criterion ready with imbalance weights

print('Pipeline ready.')

## What each class label integer means

In [8]:
# class index → class name mapping (sorted alphabetically )
processed_train = root / 'Dataset/processed/train'
if processed_train.exists():
    class_names = sorted(d.name for d in processed_train.iterdir() if d.is_dir())
    print('Label  Class name')
    print('─' * 35)
    for idx, name in enumerate(class_names):
        print(f'  {idx:2d}   {name}')

Label  Class name
───────────────────────────────────
   0   bottle-blue
   1   bottle-blue-full
   2   bottle-blue5l
   3   bottle-blue5l-full
   4   bottle-dark
   5   bottle-dark-full
   6   bottle-green
   7   bottle-green-full
   8   bottle-milk
   9   bottle-milk-full
  10   bottle-multicolor
  11   bottle-multicolorv-full
  12   bottle-oil
  13   bottle-oil-full
  14   bottle-transp
  15   bottle-transp-full
  16   bottle-yogurt
  17   canister
  18   cans
  19   detergent-box
  20   detergent-color
  21   detergent-transparent
  22   detergent-white
  23   glass-dark
  24   glass-green
  25   glass-transp
  26   juice-cardboard
  27   milk-cardboard
